# Representación de Texto en NLP — Ejemplos Prácticos

Este notebook acompaña la presentación **"Text Representation & Modelos de Lenguaje"**
y ofrece un ejemplo de código ejecutable para cada técnica de representación de texto
cubierta en el curso, organizadas según la misma taxonomía:

1. [One-Hot Encoding](#1)
2. [Bag-of-Words (BoW)](#2)
3. [TF-IDF](#3)
4. [Matrices de Co-ocurrencia](#4)
5. [Word2Vec (CBOW y Skip-gram)](#5)
6. [GloVe (vectores preentrenados)](#6)
7. [FastText (subpalabras y OOV)](#7)
8. [De palabras a frases y oraciones](#8)
9. [Embeddings contextuales (BERT)](#9)
10. [Resumen comparativo](#10)

**Referencias principales:**
- Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed., borrador).
- Zong, C., Zhao, Y. & Ma, Y. (2026). *Natural Language Processing and Large Language Models: Theory, Hands-on Codes, and Case Studies*. Tsinghua University Press / Springer.
- Manning, C. D., Raghavan, P. & Schütze, H. (2008). *Introduction to Information Retrieval*.
- Mikolov, T. et al. (2013); Pennington, J. et al. (2014); Peters, M. E. et al. (2018); Devlin, J. et al. (2019).


In [3]:
# Instalación (descomentar si hace falta en tu entorno)
# !pip install scikit-learn gensim nltk transformers torch pandas matplotlib --quiet


In [4]:
import numpy as np
import pandas as pd
from pprint import pprint

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
np.set_printoptions(precision=3, suppress=True)

print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


## Corpus de ejemplo

Usamos un pequeño corpus temático de NLP (en español) para ilustrar cada técnica.
Incluye además dos oraciones con la palabra **"banco"** (polisémica), que usaremos
más adelante para ilustrar la diferencia entre vectores estáticos y contextuales.

In [5]:
corpus = [
    "el procesamiento de lenguaje natural transforma el texto en vectores",
    "los modelos de lenguaje predicen la siguiente palabra en una secuencia",
    "las redes neuronales aprenden representaciones densas del texto",
    "el aprendizaje profundo mejora los modelos de lenguaje natural",
    "los transformers usan atencion para procesar el texto",
    "bert y gpt son modelos de lenguaje basados en transformers",
    "el texto se convierte en tokens antes de ser procesado",
    "los embeddings capturan relaciones semanticas entre palabras",
    "el modelo de lenguaje genera texto palabra por palabra",
    "las palabras similares tienen vectores similares en el espacio",
    "el aprendizaje automatico necesita datos de entrenamiento",
    "los vectores de palabras representan el significado en un espacio denso",
    "un corpus grande mejora la calidad de los embeddings",
    "las redes neuronales recurrentes procesan secuencias de texto",
    "el algoritmo de entrenamiento ajusta los pesos del modelo",
    "los datos de entrenamiento provienen de un corpus de texto",
    "fui al banco a retirar dinero para pagar la cuenta",
    "me sente en el banco del parque a leer un libro",
    "el banco central subio las tasas de interes este mes",
    "los estudiantes se sentaron en el banco de madera",
]

print(f"Tamaño del corpus: {len(corpus)} oraciones")
for i, s in enumerate(corpus[:3], 1):
    print(f"  {i}. {s}")
print("  ...")


Tamaño del corpus: 20 oraciones
  1. el procesamiento de lenguaje natural transforma el texto en vectores
  2. los modelos de lenguaje predicen la siguiente palabra en una secuencia
  3. las redes neuronales aprenden representaciones densas del texto
  ...


<a id="1"></a>
## 1. One-Hot Encoding

Cada palabra del vocabulario se representa como un vector disperso con un único 1
en la posición correspondiente a su índice, y 0 en todas las demás. No captura
orden ni semántica: cualquier par de palabras es igual de "distante" (Zong, Zhao & Ma,
2026, cap. 3.1; Jurafsky & Martin, cap. 6).

In [7]:
from sklearn.preprocessing import OneHotEncoder

# Vocabulario a partir de la primera oración
oracion = corpus[0].split()
vocab_ejemplo = sorted(set(oracion))

encoder = OneHotEncoder(sparse_output=False)
onehot = encoder.fit_transform(np.array(vocab_ejemplo).reshape(-1, 1))

df_onehot = pd.DataFrame(onehot, index=vocab_ejemplo, columns=encoder.categories_[0])
print(f"Vocabulario: {len(vocab_ejemplo)} palabras -> vectores de dimensión {onehot.shape[1]}")
df_onehot


Vocabulario: 9 palabras -> vectores de dimensión 9


,de,el,en,lenguaje,natural,procesamiento,texto,transforma,vectores
de,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
el,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
en,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
lenguaje,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
natural,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
procesamiento,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
texto,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
transforma,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
vectores,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [8]:
# Observación clave: todos los vectores son ortogonales entre sí
from numpy.linalg import norm

v1 = df_onehot.loc["texto"].values
v2 = df_onehot.loc["lenguaje"].values
similitud_coseno = np.dot(v1, v2) / (norm(v1) * norm(v2) + 1e-9)
print(f"Similitud coseno entre 'texto' y 'lenguaje': {similitud_coseno:.3f}")
print("-> Es 0: en One-Hot, dos palabras cualesquiera son igual de 'distintas'.")


Similitud coseno entre 'texto' y 'lenguaje': 0.000
-> Es 0: en One-Hot, dos palabras cualesquiera son igual de 'distintas'.


<a id="2"></a>
## 2. Bag-of-Words (BoW)

Representa cada documento como el **conteo de frecuencia** de cada palabra del
vocabulario, ignorando el orden. Es la base de la representación por frecuencia
(GeeksforGeeks, 2025).

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(corpus)

df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer_bow.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)
print(f"Matriz BoW: {df_bow.shape[0]} documentos x {df_bow.shape[1]} términos")
df_bow.iloc[:5, :10]


Matriz BoW: 20 documentos x 93 términos


,ajusta,al,algoritmo,antes,aprenden,aprendizaje,atencion,automatico,banco,basados
doc0,0,0,0,0,0,0,0,0,0,0
doc1,0,0,0,0,0,0,0,0,0,0
doc2,0,0,0,0,1,0,0,0,0,0
doc3,0,0,0,0,0,1,0,0,0,0
doc4,0,0,0,0,0,0,1,0,0,0


In [11]:
# Las palabras más frecuentes en todo el corpus
frecuencias = df_bow.sum(axis=0).sort_values(ascending=False)
print("Top 10 palabras más frecuentes:")
frecuencias.head(10)


Top 10 palabras más frecuentes:


de               16
el               13
los               9
en                8
texto             7
lenguaje          5
las               4
banco             4
un                4
entrenamiento     3
dtype: int64

<a id="3"></a>
## 3. TF-IDF

Pondera cada término por su frecuencia en el documento (**TF**) y por su rareza
en el corpus (**IDF**), penalizando palabras comunes en todos los documentos
(Manning, Raghavan & Schütze, 2008):

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out(),
    index=[f"doc{i}" for i in range(len(corpus))],
)

# Términos con mayor peso TF-IDF en el documento 0
top_terms_doc0 = df_tfidf.iloc[0].sort_values(ascending=False).head(6)
print("Términos con mayor peso TF-IDF en doc0:")
print(f'  "{corpus[0]}"\n')
top_terms_doc0


Términos con mayor peso TF-IDF en doc0:
  "el procesamiento de lenguaje natural transforma el texto en vectores"



procesamiento    0.428749
transforma       0.428749
el               0.378571
natural          0.376877
vectores         0.340073
lenguaje         0.288201
Name: doc0, dtype: float64

In [14]:
# Comparación: BoW vs TF-IDF para una palabra frecuente en todo el corpus ("texto")
# vs. una palabra distintiva de un solo documento ("bert")
comparacion = pd.DataFrame({
    "BoW (doc0)": df_bow[["texto"]].iloc[0] if "texto" in df_bow.columns else None,
    "TF-IDF (doc0)": df_tfidf[["texto"]].iloc[0] if "texto" in df_tfidf.columns else None,
})
print('TF-IDF da menos peso a "texto" (aparece en casi todos los documentos)')
print('que a un término específico como "bert" (aparece en un único documento).')
print()
print("peso TF-IDF de 'texto' en doc0:", round(df_tfidf.loc["doc0", "texto"], 3))
print("peso TF-IDF de 'bert' en doc5:  ", round(df_tfidf.loc["doc5", "bert"], 3))


TF-IDF da menos peso a "texto" (aparece en casi todos los documentos)
que a un término específico como "bert" (aparece en un único documento).

peso TF-IDF de 'texto' en doc0: 0.251
peso TF-IDF de 'bert' en doc5:   0.398


<a id="4"></a>
## 4. Matrices de Co-ocurrencia

Cuentan cuántas veces dos palabras aparecen juntas dentro de una ventana de
contexto. Codifican directamente la **hipótesis distribucional** de Firth (1957):
*"Conocerás una palabra por la compañía que frecuenta"* (Jurafsky & Martin,
cap. 6; Manning & Schütze, 1999).

In [15]:
from collections import defaultdict

def matriz_coocurrencia(corpus, ventana=2):
    """Construye una matriz de co-ocurrencia palabra-palabra con ventana simétrica."""
    vocab = sorted(set(w for doc in corpus for w in doc.split()))
    idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))

    for doc in corpus:
        tokens = doc.split()
        for i, w in enumerate(tokens):
            inicio, fin = max(0, i - ventana), min(len(tokens), i + ventana + 1)
            for j in range(inicio, fin):
                if i != j:
                    M[idx[w], idx[tokens[j]]] += 1
    return pd.DataFrame(M, index=vocab, columns=vocab)

df_coocurrencia = matriz_coocurrencia(corpus, ventana=2)
print(f"Matriz de co-ocurrencia: {df_coocurrencia.shape[0]} x {df_coocurrencia.shape[1]} (ventana=2)")

# Vista de una submatriz de palabras temáticas
palabras_interes = ["texto", "lenguaje", "modelos", "vectores", "palabra", "modelo"]
df_coocurrencia.loc[palabras_interes, palabras_interes]


Matriz de co-ocurrencia: 95 x 95 (ventana=2)


,texto,lenguaje,modelos,vectores,palabra,modelo
texto,0.0,1.0,0.0,1.0,1.0,0.0
lenguaje,1.0,0.0,3.0,0.0,0.0,1.0
modelos,0.0,3.0,0.0,0.0,0.0,0.0
vectores,1.0,0.0,0.0,0.0,0.0,0.0
palabra,1.0,0.0,0.0,0.0,2.0,0.0
modelo,0.0,1.0,0.0,0.0,0.0,0.0


In [12]:
# ¿Qué palabras co-ocurren más con "modelo"?
print("Palabras que más co-ocurren con 'modelo':")
df_coocurrencia["modelo"].sort_values(ascending=False).head(6)


Palabras que más co-ocurren con 'modelo':


el           1.0
de           1.0
del          1.0
pesos        1.0
lenguaje     1.0
algoritmo    0.0
Name: modelo, dtype: float64

<a id="5"></a>
## 5. Word2Vec — CBOW y Skip-gram

Entrenamos un modelo Word2Vec real sobre nuestro corpus con **gensim**, en sus
dos variantes (Mikolov et al., 2013a, b):

- **CBOW** (`sg=0`): predice la palabra central a partir de su contexto.
- **Skip-gram** (`sg=1`): predice el contexto a partir de la palabra central.

> ⚠️ Nuestro corpus es diminuto (solo para fines didácticos). Un Word2Vec real
> se entrena con millones de oraciones; aquí el objetivo es mostrar el
> **mecanismo**, no obtener vectores semánticamente ricos.

In [13]:
from gensim.models import Word2Vec

tokenized_corpus = [doc.split() for doc in corpus]

# CBOW
w2v_cbow = Word2Vec(
    sentences=tokenized_corpus, vector_size=50, window=3,
    min_count=1, sg=0, epochs=200, seed=42,
)

# Skip-gram
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus, vector_size=20, window=3,
    min_count=1, sg=1, epochs=30, seed=42,
)

print("Vocabulario aprendido:", len(w2v_cbow.wv.key_to_index), "palabras")
print("Dimensión del vector de 'texto':", w2v_cbow.wv["texto"].shape)
print()
print("Primeros 8 valores del vector de 'texto' (CBOW):")
print(w2v_cbow.wv["texto"][:8])


Vocabulario aprendido: 95 palabras
Dimensión del vector de 'texto': (50,)

Primeros 8 valores del vector de 'texto' (CBOW):
[ 0.051 -0.113 -0.001 -0.064 -0.055  0.237 -0.198  0.339]


In [14]:
# Palabras más similares según cada arquitectura (en nuestro corpus toy)
print("Más similares a 'lenguaje' — CBOW:")
pprint(w2v_cbow.wv.most_similar("lenguaje", topn=5))

print("\nMás similares a 'lenguaje' — Skip-gram:")
pprint(w2v_skipgram.wv.most_similar("lenguaje", topn=5))


Más similares a 'lenguaje' — CBOW:
[('el', 0.9977800846099854),
 ('banco', 0.9976662397384644),
 ('de', 0.9974484443664551),
 ('en', 0.9974260926246643),
 ('texto', 0.9971669912338257)]

Más similares a 'lenguaje' — Skip-gram:
[('natural', 0.5416581034660339),
 ('bert', 0.534976065158844),
 ('pagar', 0.5081196427345276),
 ('significado', 0.46071889996528625),
 ('profundo', 0.44134241342544556)]


**Negative sampling y hierarchical softmax.** Con vocabularios grandes,
normalizar sobre todas las palabras en cada paso de entrenamiento es costoso.
Word2Vec aproxima esto con *negative sampling* (convierte el problema en una
clasificación binaria: palabra de contexto real vs. palabras aleatorias) o con
*hierarchical softmax* (un árbol binario donde cada palabra es una hoja)
(Zong, Zhao & Ma, 2026, cap. 3.1.1). `gensim` usa negative sampling por defecto:

In [15]:
w2v_ns = Word2Vec(
    sentences=tokenized_corpus, vector_size=20, window=3, min_count=1,
    sg=1, negative=5, epochs=30, seed=42,  # negative=5 -> 5 muestras negativas por muestra positiva
)
print("Modelo entrenado con negative sampling (negative=5).")
print("Vector de 'modelo' tiene norma:", round(np.linalg.norm(w2v_ns.wv["modelo"]), 3))


Modelo entrenado con negative sampling (negative=5).
Vector de 'modelo' tiene norma: 0.134


<a id="6"></a>
## 6. GloVe — vectores preentrenados

A diferencia de Word2Vec, GloVe factoriza una **matriz de co-ocurrencia global**
del corpus (Pennington et al., 2014). Entrenar GloVe desde cero requiere un
corpus enorme, así que aquí cargamos vectores **reales, preentrenados** sobre
Wikipedia + Gigaword (`glove-wiki-gigaword-50`, 400k palabras, 50 dimensiones)
usando el downloader de `gensim`.

> Requiere conexión a internet la primera vez (~66 MB); luego queda cacheado
> localmente.

In [16]:
import gensim.downloader as api

try:
    glove = api.load("glove-wiki-gigaword-50")
    print(f"GloVe cargado: {len(glove)} palabras, vectores de {glove.vector_size} dimensiones")
    glove_disponible = True
except Exception as e:
    print("No se pudo descargar GloVe (¿sin conexión a internet?).")
    print("Detalle:", type(e).__name__, str(e)[:200])
    glove_disponible = False


[=================================================-] 98.6% 65.0/66.0MB downloadedGloVe cargado: 400000 palabras, vectores de 50 dimensiones


In [17]:
if glove_disponible:
    print("Palabras más similares a 'language':")
    pprint(glove.most_similar("language", topn=5))

    print("\nAnalogía clásica: king - man + woman ≈ ?")
    pprint(glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))

    print("\nAnalogía: paris - france + spain ≈ ?")
    pprint(glove.most_similar(positive=["paris", "spain"], negative=["france"], topn=3))


Palabras más similares a 'language':
[('languages', 0.8814865946769714),
 ('word', 0.8100197315216064),
 ('spoken', 0.8074647784233093),
 ('vocabulary', 0.7903235554695129),
 ('translation', 0.7879166007041931)]

Analogía clásica: king - man + woman ≈ ?
[('queen', 0.8523604869842529),
 ('throne', 0.7664334177970886),
 ('prince', 0.759214460849762)]

Analogía: paris - france + spain ≈ ?
[('aires', 0.8409486413002014),
 ('buenos', 0.836663007736206),
 ('madrid', 0.8128822445869446)]


Esto es justamente lo que el capítulo 3 de Zong, Zhao & Ma (2026) describe
como la propiedad más célebre de los embeddings estáticos aprendidos por
predicción: **la aritmética vectorial refleja relaciones semánticas**
(*E(King) − E(Man) + E(Woman) ≈ E(Queen)*).

<a id="7"></a>
## 7. FastText — subpalabras y palabras fuera de vocabulario (OOV)

FastText representa cada palabra como una **bolsa de n-gramas de caracteres**,
además del token completo. Esto le permite construir un vector razonable
incluso para palabras que **nunca vio durante el entrenamiento** — algo
imposible para Word2Vec o GloVe estándar (Joulin et al., 2017).

In [18]:
from gensim.models import FastText

ft_model = FastText(
    sentences=tokenized_corpus, vector_size=20, window=3,
    min_count=1, sg=1, epochs=30, min_n=3, max_n=6, seed=42,
)

palabra_oov = "lenguajes"  # variante de "lenguaje" que NO aparece en el corpus
print(f"¿'{palabra_oov}' está en el vocabulario? ", palabra_oov in ft_model.wv.key_to_index)

try:
    vector_oov = ft_model.wv[palabra_oov]
    print(f"FastText igual genera un vector para '{palabra_oov}' (vía subpalabras).")
    print("Similitud con 'lenguaje':", round(ft_model.wv.similarity(palabra_oov, "lenguaje"), 3))
except KeyError:
    print("No se pudo generar el vector.")


¿'lenguajes' está en el vocabulario?  False
FastText igual genera un vector para 'lenguajes' (vía subpalabras).
Similitud con 'lenguaje': 0.851


<a id="8"></a>
## 8. De palabras a frases y oraciones

Un embedding de palabra no alcanza para representar una oración completa.
Las estrategias más simples combinan los vectores de sus palabras
(Zong, Zhao & Ma, 2026, cap. 3.2–3.3):

- **Promedio (average pooling):** $\displaystyle ph_N = \frac{1}{N}\sum_{k=1}^{N} x(w_k)$
- **Máximo (max pooling):** $\displaystyle ph_N = \max_k \big(x(w_1)_k, \dots, x(w_N)_k\big)$
- **Promedio ponderado:** igual que el promedio, pero ponderando cada palabra
  (p. ej. por su peso TF-IDF).

In [19]:
def vector_oracion(oracion, modelo, metodo="promedio"):
    tokens = [t for t in oracion.split() if t in modelo.wv.key_to_index]
    if not tokens:
        return np.zeros(modelo.vector_size)
    vectores = np.array([modelo.wv[t] for t in tokens])
    if metodo == "promedio":
        return vectores.mean(axis=0)
    elif metodo == "maximo":
        return vectores.max(axis=0)
    else:
        raise ValueError("metodo debe ser 'promedio' o 'maximo'")

oracion_a = "el modelo de lenguaje genera texto"
oracion_b = "los transformers procesan el texto"
oracion_c = "me sente en el banco del parque"

vec_a = vector_oracion(oracion_a, w2v_skipgram, "promedio")
vec_b = vector_oracion(oracion_b, w2v_skipgram, "promedio")
vec_c = vector_oracion(oracion_c, w2v_skipgram, "promedio")

def sim_coseno(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)

print(f"sim('{oracion_a}',\n    '{oracion_b}') = {sim_coseno(vec_a, vec_b):.3f}  (ambas hablan de NLP)")
print(f"sim('{oracion_a}',\n    '{oracion_c}') = {sim_coseno(vec_a, vec_c):.3f}  (temas distintos)")


sim('el modelo de lenguaje genera texto',
    'los transformers procesan el texto') = 0.575  (ambas hablan de NLP)
sim('el modelo de lenguaje genera texto',
    'me sente en el banco del parque') = 0.394  (temas distintos)


<a id="9"></a>
## 9. Embeddings contextuales (BERT) — vectores estáticos vs. dinámicos

Hasta aquí, cada palabra tiene **un único vector fijo**, sin importar el
contexto: son vectores **estáticos**. Los modelos contextuales (ELMo, BERT,
GPT) calculan el vector de cada palabra como función de **toda la oración**,
resolviendo el problema de la polisemia (Jurafsky & Martin, cap. 11;
Zong, Zhao & Ma, 2026, cap. 3.1.2 y cap. 6).

Ejemplo clásico: la palabra **"banco"** en:
- *"Fui al banco a retirar dinero."* → sentido financiero
- *"Me senté en el banco del parque."* → sentido de asiento

Con Word2Vec/GloVe, "banco" tiene el mismo vector en ambas oraciones. Con BERT,
tendría dos vectores distintos.

El siguiente bloque usa `transformers` para calcular embeddings contextuales
reales con BERT. **Requiere internet** para descargar el modelo preentrenado
(~440 MB) la primera vez. Si no hay conexión disponible, el notebook cae a una
simulación simplificada que solo ilustra la idea, sin ser BERT real.

In [20]:
USAR_BERT_REAL = True
bert_disponible = False

if USAR_BERT_REAL:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModel

        modelo_bert = "bert-base-multilingual-cased"  # soporta español
        tokenizer = AutoTokenizer.from_pretrained(modelo_bert)
        model = AutoModel.from_pretrained(modelo_bert)
        model.eval()
        bert_disponible = True
        print(f"BERT ('{modelo_bert}') cargado correctamente.")
    except Exception as e:
        print("No se pudo cargar BERT (probablemente sin acceso a internet).")
        print("Detalle:", type(e).__name__, str(e)[:200])
        print()
        print("-> Se usará una SIMULACIÓN simplificada más abajo, solo con fines ilustrativos.")


No se pudo cargar BERT (probablemente sin acceso a internet).
Detalle: ModuleNotFoundError No module named 'torch'

-> Se usará una SIMULACIÓN simplificada más abajo, solo con fines ilustrativos.


In [21]:
def vector_contextual_bert(oracion, palabra_objetivo):
    """Devuelve el embedding contextual (última capa oculta) de `palabra_objetivo`
    dentro de `oracion`, usando BERT real."""
    inputs = tokenizer(oracion, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        outputs = model(**inputs)
    ultima_capa = outputs.last_hidden_state[0]  # [n_tokens, hidden_size]

    # Localizamos el (sub)token de la palabra objetivo (búsqueda simple, no exhaustiva)
    coincidencias = [i for i, t in enumerate(tokens) if palabra_objetivo.lower() in t.lower()]
    if not coincidencias:
        return None, tokens
    return ultima_capa[coincidencias[0]].numpy(), tokens


if bert_disponible:
    oracion_1 = "Fui al banco a retirar dinero."
    oracion_2 = "Me senté en el banco del parque."

    vec_banco_1, tok1 = vector_contextual_bert(oracion_1, "banco")
    vec_banco_2, tok2 = vector_contextual_bert(oracion_2, "banco")

    similitud = sim_coseno(vec_banco_1, vec_banco_2)
    print(f'Tokens oración 1: {tok1}')
    print(f'Tokens oración 2: {tok2}')
    print()
    print(f"Similitud coseno entre los dos vectores de 'banco': {similitud:.3f}")
    print("-> Al ser BERT contextual, esperamos una similitud NO cercana a 1.0,")
    print("   porque el modelo distingue el sentido financiero del sentido 'asiento'.")


In [22]:
if not bert_disponible:
    # --- SIMULACION SIMPLIFICADA (NO es BERT real) ---
    # Sin acceso a un modelo contextual real, ilustramos la idea con un truco
    # deliberadamente simple: representamos el "contexto" de banco en cada oracion
    # como un vector Bag-of-Words de las palabras que lo rodean. Como el contexto
    # de palabras es distinto en cada oracion, el vector resultante tambien lo es
    # -- esa es, en esencia, la intuicion detras de un embedding contextual.
    print("Simulacion simplificada de contextualizacion (NO usa BERT real):\n")

    from sklearn.feature_extraction.text import CountVectorizer

    oracion_1 = "fui al banco a retirar dinero"
    oracion_2 = "me sente en el banco del parque"

    contexto_1 = oracion_1.replace("banco", "").split()
    contexto_2 = oracion_2.replace("banco", "").split()

    cv = CountVectorizer()
    cv.fit([" ".join(contexto_1 + contexto_2)])
    v1 = cv.transform([" ".join(contexto_1)]).toarray()[0]
    v2 = cv.transform([" ".join(contexto_2)]).toarray()[0]

    print(f"Similitud (simulada) entre los dos contextos de 'banco': {sim_coseno(v1, v2):.3f}")
    print("(Un valor bajo ilustra la idea de que cada oracion aporta un contexto distinto,")
    print(" y por lo tanto un modelo contextual real generaria un vector distinto para")
    print(" 'banco' en cada una. Esto es solo una aproximacion pedagogica con BoW,")
    print(" no el computo real de BERT.)")


Simulacion simplificada de contextualizacion (NO usa BERT real):

Similitud (simulada) entre los dos contextos de 'banco': 0.000
(Un valor bajo ilustra la idea de que cada oracion aporta un contexto distinto,
 y por lo tanto un modelo contextual real generaria un vector distinto para
 'banco' en cada una. Esto es solo una aproximacion pedagogica con BoW,
 no el computo real de BERT.)


**Para usar BERT real fuera de este entorno:** basta con conexión a internet;
`transformers` descarga el modelo automáticamente la primera vez y lo cachea
localmente. El resto del código no cambia.

<a id="10"></a>
## 10. Resumen comparativo

In [23]:
resumen = pd.DataFrame([
    ["One-Hot Encoding",        "Frequency-based", "Disperso",  "Alta (|V|)",  "No"],
    ["Bag-of-Words",            "Frequency-based", "Disperso",  "Alta (|V|)",  "No"],
    ["TF-IDF",                  "Frequency-based", "Disperso",  "Alta (|V|)",  "No"],
    ["Co-ocurrencia",           "Frequency-based", "Disperso",  "Alta (|V|)",  "No"],
    ["Word2Vec",                "Prediction-based","Denso",     "Baja (50-300)","No"],
    ["GloVe",                   "Prediction-based","Denso",     "Baja (50-300)","No"],
    ["FastText",                "Prediction-based","Denso",     "Baja (50-300)","No (pero sí para OOV)"],
    ["ELMo / BERT / GPT",       "Contextualized",  "Denso",     "Media (768+)", "Sí"],
], columns=["Técnica", "Familia", "Tipo de vector", "Dimensionalidad", "¿Contextual?"])

resumen


,Técnica,Familia,Tipo de vector,Dimensionalidad,¿Contextual?
0,One-Hot Encoding,Frequency-based,Disperso,Alta (|V|),No
1,Bag-of-Words,Frequency-based,Disperso,Alta (|V|),No
2,TF-IDF,Frequency-based,Disperso,Alta (|V|),No
3,Co-ocurrencia,Frequency-based,Disperso,Alta (|V|),No
4,Word2Vec,Prediction-based,Denso,Baja (50-300),No
5,GloVe,Prediction-based,Denso,Baja (50-300),No
6,FastText,Prediction-based,Denso,Baja (50-300),No (pero sí para OOV)
7,ELMo / BERT / GPT,Contextualized,Denso,Media (768+),Sí


---
### Notas finales

- Todos los ejemplos de esta libreta usan un corpus diminuto con fines
  **didácticos**: ilustran el mecanismo de cada técnica, no su desempeño real.
  Para resultados semánticamente ricos con Word2Vec o FastText hace falta
  entrenar sobre corpus de millones de oraciones.
- GloVe y BERT sí se cargaron con **pesos preentrenados reales** (cuando hay
  conexión a internet), por lo que sus resultados (analogías, similitud
  contextual) sí reflejan relaciones semánticas genuinas del inglés.
- Para producción, las librerías recomendadas son: `scikit-learn` (BoW,
  TF-IDF), `gensim` (Word2Vec, GloVe, FastText) y `transformers` de
  Hugging Face (BERT, GPT y modelos más recientes).
